# BDC Statistics Explore 2026 — SigLIP2 Conditional Severity, 4×GPU — FixRes Loader Fix

**Experiment 2** keeps the original modeling recipe unchanged and only improves compute orchestration:

- `google/siglip2-base-patch16-384`
- aspect-ratio-preserving resize + pad to 384
- shared disaster head + 3 disaster-conditioned severity heads
- soft severity routing at inference
- Stage 1: frozen vision backbone, train heads only
- Stage 2: unfreeze only the final 4 vision transformer blocks + final norm/pooling + heads
- CE loss: `0.35 × disaster + 0.65 × severity`
- mild photometric augmentation + horizontal flip only
- TRAIN-only exact/strict-near grouping with 5-fold `StratifiedGroupKFold`
- primary selection metric: pooled OOF row-level **Micro F1**

## 4-GPU execution strategy

1. Build the TRAIN manifest, duplicate groups, and 5 folds once on CPU.
2. Run folds 0–3 independently on GPUs 0–3; run fold 4 on the first available GPU.
3. Pool all 5 OOF predictions to select partial-FT epoch and flip-TTA using TRAIN only.
4. Retrain one final model on 100% TRAIN with 4-GPU DDP.
5. Keep final effective/global batch size at 32: `8 per GPU × 4 GPUs`.
6. Run TEST inference only after TRAIN-only model selection is complete.

No TEST-derived calibration, pseudo-labeling, filename-range rules, TEST hash lookup, or TEST-target logic is used.

## Expected dataset paths

```text
/workspace/dataset/SE/TRAIN
/workspace/dataset/SE/TEST
/workspace/dataset/SE/TRAIN/Solution.csv
```


In [1]:
!nvidia-smi

Thu Sep  3 02:59:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.64.03              Driver Version: 575.64.03      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:98:00.0 Off |                  N/A |
|  0%   34C    P8              3W /  575W |       2MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install lightweight dependencies only. Keep the Vast.ai PyTorch/CUDA build intact.
%pip install -q -U "transformers==4.57.1" "accelerate>=1.10.0" scikit-learn pandas pillow imagehash tqdm safetensors


Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import os, sys, json, subprocess, time, shlex, textwrap

TRAIN_DIR = Path('/workspace/dataset/SE/TRAIN')
TEST_DIR = Path('/workspace/dataset/SE/TEST')
SOLUTION_PATH = Path('/workspace/dataset/SE/TRAIN/Solution.csv')
OUTPUT_DIR = Path('/workspace/output/siglip2_b384_condsev_4gpu')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_DIR.is_dir(), f'Missing TRAIN_DIR: {TRAIN_DIR}'
assert TEST_DIR.is_dir(), f'Missing TEST_DIR: {TEST_DIR}'
assert SOLUTION_PATH.is_file(), f'Missing SOLUTION_PATH: {SOLUTION_PATH}'
print('TRAIN:', TRAIN_DIR)
print('TEST :', TEST_DIR)
print('SOLUTION:', SOLUTION_PATH)
print('OUTPUT:', OUTPUT_DIR)

TRAIN: /workspace/dataset/SE/TRAIN
TEST : /workspace/dataset/SE/TEST
SOLUTION: /workspace/dataset/SE/TRAIN/Solution.csv
OUTPUT: /workspace/output/siglip2_b384_condsev_4gpu


## Runtime preflight

RTX 5090 requires a PyTorch build that supports Blackwell. This cell checks CUDA availability and visible GPUs before any expensive preprocessing/training starts.

In [4]:
import torch
print('torch:', torch.__version__)
print('torch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('visible GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, f'{p.total_memory/1024**3:.1f} GB', 'capability', torch.cuda.get_device_capability(i))
assert torch.cuda.is_available(), 'CUDA is required for this notebook.'
assert torch.cuda.device_count() >= 4, 'This 4-GPU notebook expects at least 4 visible CUDA GPUs.'

torch: 2.11.0+cu128
torch CUDA build: 12.8
CUDA available: True
visible GPUs: 4
0 NVIDIA GeForce RTX 5090 31.4 GB capability (12, 0)
1 NVIDIA GeForce RTX 5090 31.4 GB capability (12, 0)
2 NVIDIA GeForce RTX 5090 31.4 GB capability (12, 0)
3 NVIDIA GeForce RTX 5090 31.4 GB capability (12, 0)


## Prefetch SigLIP2 checkpoint once

This prevents four parallel CV workers from trying to download the same model simultaneously.

In [5]:
from huggingface_hub import snapshot_download
MODEL_ID = 'google/siglip2-base-patch16-384'
cache_path = snapshot_download(MODEL_ID)
print('Model cached at:', cache_path)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/276 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Model cached at: /root/.cache/huggingface/hub/models--google--siglip2-base-patch16-384/snapshots/f775b65a79762255128c981547af89addcfe0f88


## Build one immutable TRAIN-only manifest and grouped folds

Grouping uses exact SHA-256 plus a conservative perceptual graph (`pHash≤1`, `dHash≤2`, `aHash≤4`). All members of a connected component stay in the same validation fold.

In [6]:
import hashlib, math
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import imagehash
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import StratifiedGroupKFold

SEED = 20260901
IMAGE_EXTS = {'.jpg','.jpeg','.png','.jfif','.bmp','.webp'}
JENIS = ['BANJIR','GEMPA BUMI','KEBAKARAN']
KERUSAKAN = ['KERUSAKAN RINGAN','KERUSAKAN SEDANG','KERUSAKAN BERAT']
JENIS_TO_IDX = {x:i for i,x in enumerate(JENIS)}
KERUSAKAN_TO_IDX = {x:i for i,x in enumerate(KERUSAKAN)}

def enumerate_train(root):
    rows=[]
    for jenis in JENIS:
        for ker in KERUSAKAN:
            d=root/jenis/ker
            assert d.is_dir(), f'Missing class directory: {d}'
            for p in sorted(d.iterdir()):
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                    rows.append({
                        'path':str(p), 'jenis':jenis, 'kerusakan':ker,
                        'jenis_idx':JENIS_TO_IDX[jenis],
                        'kerusakan_idx':KERUSAKAN_TO_IDX[ker],
                        'joint_idx':JENIS_TO_IDX[jenis]*3+KERUSAKAN_TO_IDX[ker],
                    })
    return pd.DataFrame(rows)

train_df=enumerate_train(TRAIN_DIR)
print('TRAIN images:',len(train_df))
display(train_df.groupby(['jenis','kerusakan']).size().unstack(fill_value=0))

HASH_CACHE = OUTPUT_DIR/'train_hashes.csv'
GROUP_CACHE = OUTPUT_DIR/'train_manifest_5fold.csv'

def hash_one(path):
    p=Path(path)
    sha=hashlib.sha256(p.read_bytes()).hexdigest()
    with Image.open(p) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        return sha, int(str(imagehash.phash(im)),16), int(str(imagehash.dhash(im)),16), int(str(imagehash.average_hash(im)),16)

if HASH_CACHE.exists():
    hdf=pd.read_csv(HASH_CACHE)
    print('Loaded hash cache:',HASH_CACHE)
else:
    with ThreadPoolExecutor(max_workers=min(32, os.cpu_count() or 8)) as ex:
        vals=list(__import__('tqdm').tqdm(ex.map(hash_one,train_df.path), total=len(train_df), desc='Hashing TRAIN'))
    hdf=pd.DataFrame(vals,columns=['sha256','phash','dhash','ahash'])
    hdf.insert(0,'path',train_df.path.values)
    hdf.to_csv(HASH_CACHE,index=False)
    print('Saved hash cache:',HASH_CACHE)

assert list(hdf.path)==list(train_df.path), 'Hash cache path order mismatch.'

class DSU:
    def __init__(self,n): self.p=list(range(n)); self.r=[0]*n
    def find(self,x):
        while self.p[x]!=x:
            self.p[x]=self.p[self.p[x]]; x=self.p[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.r[a]<self.r[b]:a,b=b,a
        self.p[b]=a
        if self.r[a]==self.r[b]:self.r[a]+=1

def ham(a,b): return (int(a)^int(b)).bit_count()

def build_groups(hdf):
    n=len(hdf); dsu=DSU(n)
    sha_map={}
    for i,s in enumerate(hdf.sha256):
        if s in sha_map: dsu.union(i,sha_map[s])
        else: sha_map[s]=i
    # Candidate generation from four 16-bit pHash bands, matching the original TRAIN-only recipe.
    buckets={}; cand=set()
    ph=hdf.phash.to_numpy()
    for i,val in enumerate(ph):
        v=int(val)
        for band in range(4):
            key=(band,(v>>(16*band))&0xFFFF)
            for j in buckets.get(key,[]): cand.add((j,i))
            buckets.setdefault(key,[]).append(i)
    dh=hdf.dhash.to_numpy(); ah=hdf.ahash.to_numpy()
    kept=0
    for a,b in __import__('tqdm').tqdm(cand,desc='Strict near-duplicate edges'):
        if ham(ph[a],ph[b])<=1 and ham(dh[a],dh[b])<=2 and ham(ah[a],ah[b])<=4:
            dsu.union(a,b); kept+=1
    roots=[dsu.find(i) for i in range(n)]
    remap={r:j for j,r in enumerate(sorted(set(roots)))}
    return np.array([remap[r] for r in roots]), kept

if GROUP_CACHE.exists():
    manifest=pd.read_csv(GROUP_CACHE)
    print('Loaded grouped-fold manifest:',GROUP_CACHE)
else:
    scene_group,near_edges=build_groups(hdf)
    manifest=train_df.copy()
    manifest['scene_group']=scene_group
    manifest['fold']=-1
    sgkf=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
    for fold,(_,va) in enumerate(sgkf.split(manifest,manifest.joint_idx,groups=manifest.scene_group)):
        manifest.loc[va,'fold']=fold
    assert (manifest.fold>=0).all()
    manifest.to_csv(GROUP_CACHE,index=False)
    print('strict near edges:',near_edges)
    print('Saved grouped-fold manifest:',GROUP_CACHE)

assert len(manifest)==len(train_df)
assert manifest.path.is_unique
for f in range(5):
    va=set(manifest.loc[manifest.fold==f,'scene_group'])
    tr=set(manifest.loc[manifest.fold!=f,'scene_group'])
    assert not (va & tr), f'Group leakage in fold {f}'
print(manifest.groupby(['fold','joint_idx']).size().unstack(fill_value=0))

TRAIN images: 17482


kerusakan,KERUSAKAN BERAT,KERUSAKAN RINGAN,KERUSAKAN SEDANG
jenis,,,
BANJIR,1968,2018,1971
GEMPA BUMI,1623,1393,2730
KEBAKARAN,2025,1746,2008


Hashing TRAIN: 100%|██████████| 17482/17482 [03:39<00:00, 79.52it/s] 


Saved hash cache: /workspace/output/siglip2_b384_condsev_4gpu/train_hashes.csv


Strict near-duplicate edges: 100%|██████████| 152663/152663 [00:00<00:00, 1054285.51it/s]


strict near edges: 4111
Saved grouped-fold manifest: /workspace/output/siglip2_b384_condsev_4gpu/train_manifest_5fold.csv
joint_idx    0    1    2    3    4    5    6    7    8
fold                                                  
0          404  394  393  279  546  325  349  402  405
1          403  394  394  279  546  324  349  402  405
2          404  394  394  279  546  324  350  401  405
3          404  395  393  278  546  325  349  401  405
4          403  394  394  278  546  325  349  402  405


## Write the reusable single-GPU / DDP training runner

The runner is launched as independent subprocesses for CV folds and through `torchrun` for final 4-GPU DDP training. This avoids notebook multiprocessing/CUDA-state pitfalls.

In [7]:
RUNNER_PATH = OUTPUT_DIR / 'train_runner.py'
RUNNER_CODE = r'''#!/usr/bin/env python
import os, sys, json, math, time, random, argparse, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from transformers import SiglipVisionModel, AutoImageProcessor
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm

TRAIN_DIR=Path('/workspace/dataset/SE/TRAIN')
TEST_DIR=Path('/workspace/dataset/SE/TEST')
SOLUTION_PATH=Path('/workspace/dataset/SE/TRAIN/Solution.csv')
OUT=Path('/workspace/output/siglip2_b384_condsev_4gpu')
MANIFEST=OUT/'train_manifest_5fold.csv'
MODEL_ID='google/siglip2-base-patch16-384'
SEED=20260901
IMAGE_SIZE=384
PAD_RGB=(128,128,128)
DROPOUT=0.15
UNFREEZE_LAST_N=4
CV_BATCH=32
FINAL_GLOBAL_BATCH=32
EVAL_BATCH=64
NUM_WORKERS=12
HEAD_EPOCHS=2
MAX_PARTIAL_EPOCHS=5
HEAD_LR=7e-4
PARTIAL_HEAD_LR=1e-4
PARTIAL_BACKBONE_LR=1e-5
WEIGHT_DECAY=0.05
WARMUP_RATIO=0.08
GRAD_CLIP=1.0
W_J=0.35
W_K=0.65
HFLIP_P=0.50
BRIGHTNESS=0.12
CONTRAST=0.12
SATURATION=0.08
JENIS=['BANJIR','GEMPA BUMI','KEBAKARAN']
KERUSAKAN=['KERUSAKAN RINGAN','KERUSAKAN SEDANG','KERUSAKAN BERAT']
IDX_TO_JENIS={i:x for i,x in enumerate(JENIS)}
IDX_TO_KER={i:x for i,x in enumerate(KERUSAKAN)}
SUB_JENIS={'BANJIR':1,'GEMPA BUMI':2,'KEBAKARAN':3}
# Explicit submission encoding. Keep this fixed and documented; never infer it from TEST predictions.
SUB_KER={'KERUSAKAN BERAT':1,'KERUSAKAN RINGAN':2,'KERUSAKAN SEDANG':3}

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=True

def row_micro(yj,pk,yjhat,pkhat):
    yt=np.concatenate([np.asarray(yj),np.asarray(pk)])
    yp=np.concatenate([np.asarray(yjhat),np.asarray(pkhat)])
    return float(f1_score(yt,yp,average='micro'))

def metrics(yj,yk,pj,pk):
    return {'micro_f1':row_micro(yj,yk,pj,pk),
            'acc_jenis':float(accuracy_score(yj,pj)),
            'acc_kerusakan':float(accuracy_score(yk,pk))}

processor=AutoImageProcessor.from_pretrained(MODEL_ID, local_files_only=True)
mean=np.array(processor.image_mean,dtype=np.float32).reshape(3,1,1)
std=np.array(processor.image_std,dtype=np.float32).reshape(3,1,1)

def decode(path,train=False,force_flip=False):
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        if train:
            if random.random()<HFLIP_P: im=ImageOps.mirror(im)
            if BRIGHTNESS>0: im=ImageEnhance.Brightness(im).enhance(1+random.uniform(-BRIGHTNESS,BRIGHTNESS))
            if CONTRAST>0: im=ImageEnhance.Contrast(im).enhance(1+random.uniform(-CONTRAST,CONTRAST))
            if SATURATION>0: im=ImageEnhance.Color(im).enhance(1+random.uniform(-SATURATION,SATURATION))
        elif force_flip:
            im=ImageOps.mirror(im)
        w,h=im.size
        scale=min(IMAGE_SIZE/w,IMAGE_SIZE/h)
        nw=max(1,round(w*scale)); nh=max(1,round(h*scale))
        im=im.resize((nw,nh),Image.Resampling.BICUBIC)
        canvas=Image.new('RGB',(IMAGE_SIZE,IMAGE_SIZE),PAD_RGB)
        canvas.paste(im,((IMAGE_SIZE-nw)//2,(IMAGE_SIZE-nh)//2))
        arr=np.asarray(canvas,dtype=np.float32).transpose(2,0,1)/255.0
        arr=(arr-mean)/std
        return torch.from_numpy(arr)

class ImgDS(Dataset):
    def __init__(self,df,train=False,labelled=True,flip=False):
        self.df=df.reset_index(drop=True); self.train=train; self.labelled=labelled; self.flip=flip
    def __len__(self):return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; x=decode(r.path,self.train,self.flip)
        out={'pixel_values':x,'row_index':int(r.row_index) if 'row_index' in self.df.columns else i}
        if self.labelled:
            out['jenis_idx']=int(r.jenis_idx); out['kerusakan_idx']=int(r.kerusakan_idx)
        return out

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision=SiglipVisionModel.from_pretrained(MODEL_ID, local_files_only=True)
        d=self.vision.config.hidden_size
        self.feature_norm=nn.LayerNorm(d)
        self.drop=nn.Dropout(DROPOUT)
        self.disaster=nn.Linear(d,3)
        self.severity=nn.ModuleList([nn.Linear(d,3) for _ in range(3)])
    def features(self,pixel_values):
        o=self.vision(pixel_values=pixel_values,return_dict=True)
        z=o.pooler_output if getattr(o,'pooler_output',None) is not None else o.last_hidden_state.mean(1)
        return self.drop(self.feature_norm(z.float()))
    def forward(self,pixel_values):
        z=self.features(pixel_values)
        j=self.disaster(z)
        cond=torch.stack([h(z) for h in self.severity],dim=1) # B,3,3
        jp=F.softmax(j,dim=-1)
        kp=torch.sum(jp.unsqueeze(-1)*F.softmax(cond,dim=-1),dim=1)
        return j,cond,kp

def set_stage(m,stage):
    for p in m.vision.parameters():p.requires_grad=False
    for p in m.feature_norm.parameters():p.requires_grad=True
    for p in m.disaster.parameters():p.requires_grad=True
    for h in m.severity:
        for p in h.parameters():p.requires_grad=True
    if stage=='partial':
        core=getattr(m.vision,'vision_model',m.vision)
        if hasattr(core,'encoder') and hasattr(core.encoder,'layers'): layers=core.encoder.layers
        elif hasattr(m.vision,'encoder') and hasattr(m.vision.encoder,'layers'): layers=m.vision.encoder.layers
        else: raise AttributeError('Could not locate SigLIP2 transformer layers')
        for layer in layers[-UNFREEZE_LAST_N:]:
            for p in layer.parameters():p.requires_grad=True
        for attr in ('post_layernorm','head'):
            module=getattr(core,attr,None)
            if module is not None:
                for p in module.parameters():p.requires_grad=True

def make_optimizer(m,stage):
    head=[]; back=[]
    for name,p in m.named_parameters():
        if not p.requires_grad:continue
        (back if name.startswith('vision.') else head).append(p)
    groups=[]
    if back:groups.append({'params':back,'lr':PARTIAL_BACKBONE_LR if stage=='partial' else HEAD_LR})
    if head:groups.append({'params':head,'lr':PARTIAL_HEAD_LR if stage=='partial' else HEAD_LR})
    return torch.optim.AdamW(groups,weight_decay=WEIGHT_DECAY)

def cosine_sched(opt,total_steps):
    warm=max(1,int(total_steps*WARMUP_RATIO))
    def fn(step):
        if step<warm:return max(1e-8,step/warm)
        prog=(step-warm)/max(1,total_steps-warm)
        return 0.5*(1+math.cos(math.pi*min(1,prog)))
    return torch.optim.lr_scheduler.LambdaLR(opt,fn)

def loss_fn(j,cond,yj,yk):
    lj=F.cross_entropy(j,yj)
    chosen=cond[torch.arange(len(yj),device=yj.device),yj]
    lk=F.cross_entropy(chosen,yk)
    return W_J*lj+W_K*lk

def loader(df,train,labelled,batch,sampler=None,flip=False):
    return DataLoader(ImgDS(df,train,labelled,flip),batch_size=batch,shuffle=(train and sampler is None),
        sampler=sampler,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=(NUM_WORKERS>0),drop_last=False)

def train_epoch(model,ld,opt,sch,device,distributed=False):
    model.train(); total=0.; n=0
    pbar=tqdm(ld,disable=(distributed and dist.get_rank()!=0),leave=False)
    for b in pbar:
        x=b['pixel_values'].to(device,non_blocking=True); yj=b['jenis_idx'].to(device); yk=b['kerusakan_idx'].to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.bfloat16):
            j,c,_=model(x); loss=loss_fn(j,c,yj,yk)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); opt.step(); sch.step()
        total+=float(loss.detach())*len(x); n+=len(x)
    if distributed:
        t=torch.tensor([total,n],device=device,dtype=torch.float64); dist.all_reduce(t,op=dist.ReduceOp.SUM); total,n=t.tolist()
    return total/max(1,n)

@torch.no_grad()
def predict(model,ld,device,distributed=False):
    model.eval(); rows=[]; yj=[]; yk=[]; pj=[]; pk=[]; pjp=[]; pkp=[]
    for b in tqdm(ld,disable=(distributed and dist.get_rank()!=0),leave=False):
        x=b['pixel_values'].to(device,non_blocking=True)
        with torch.autocast('cuda',dtype=torch.bfloat16): j,c,kprob=model(x)
        jp=F.softmax(j,dim=-1)
        rows.extend(np.asarray(b['row_index']).tolist()); pjp.append(jp.float().cpu().numpy()); pkp.append(kprob.float().cpu().numpy())
        pj.extend(jp.argmax(-1).cpu().tolist()); pk.extend(kprob.argmax(-1).cpu().tolist())
        if 'jenis_idx' in b:
            yj.extend(np.asarray(b['jenis_idx']).tolist()); yk.extend(np.asarray(b['kerusakan_idx']).tolist())
    return {'row_index':np.asarray(rows),'yj':np.asarray(yj),'yk':np.asarray(yk),'pj':np.asarray(pj),'pk':np.asarray(pk),
            'pj_prob':np.concatenate(pjp),'pk_prob':np.concatenate(pkp)}

def blend(a,b):
    return {'row_index':a['row_index'],'yj':a['yj'],'yk':a['yk'],
            'pj_prob':(a['pj_prob']+b['pj_prob'])/2,'pk_prob':(a['pk_prob']+b['pk_prob'])/2}

def fit_fold(fold):
    seed_all(SEED+fold)
    df=pd.read_csv(MANIFEST); df['row_index']=np.arange(len(df))
    tr=df[df.fold!=fold].reset_index(drop=True); va=df[df.fold==fold].reset_index(drop=True)
    device=torch.device('cuda:0')
    model=Model().to(device)
    fold_dir=OUT/f'fold_{fold}'; fold_dir.mkdir(parents=True,exist_ok=True)
    best=-1; best_meta=None; best_path=fold_dir/'best.pt'
    history=[]
    stages=[('heads',HEAD_EPOCHS),('partial',MAX_PARTIAL_EPOCHS)]
    for stage,n_epochs in stages:
        set_stage(model,stage)
        opt=make_optimizer(model,stage)
        trld=loader(tr,True,True,CV_BATCH)
        vald=loader(va,False,True,EVAL_BATCH)
        sch=cosine_sched(opt,len(trld)*n_epochs)
        for ep in range(1,n_epochs+1):
            t=time.time(); loss=train_epoch(model,trld,opt,sch,device)
            pr=predict(model,vald,device); mm=metrics(pr['yj'],pr['yk'],pr['pj'],pr['pk'])
            rec={'stage':stage,'stage_epoch':ep,'loss':loss,**mm}; history.append(rec)
            print(f'FOLD {fold} {stage} {ep}/{n_epochs} loss={loss:.4f} micro={mm["micro_f1"]:.6f} J={mm["acc_jenis"]:.5f} K={mm["acc_kerusakan"]:.5f} min={(time.time()-t)/60:.1f}',flush=True)
            if mm['micro_f1']>best:
                best=mm['micro_f1']; best_meta={'stage':stage,'stage_epoch':ep,'micro_f1':best}
                torch.save({'model':model.state_dict(),'meta':best_meta},best_path)
    ck=torch.load(best_path,map_location=device); model.load_state_dict(ck['model'])
    val_plain=loader(va,False,True,EVAL_BATCH,flip=False); val_flip=loader(va,False,True,EVAL_BATCH,flip=True)
    p0=predict(model,val_plain,device); p1=predict(model,val_flip,device); pt=blend(p0,p1)
    plain_m=metrics(p0['yj'],p0['yk'],p0['pj_prob'].argmax(1),p0['pk_prob'].argmax(1))
    tta_m=metrics(pt['yj'],pt['yk'],pt['pj_prob'].argmax(1),pt['pk_prob'].argmax(1))
    np.savez_compressed(fold_dir/'oof.npz',row_index=p0['row_index'],yj=p0['yj'],yk=p0['yk'],plain_j=p0['pj_prob'],plain_k=p0['pk_prob'],tta_j=pt['pj_prob'],tta_k=pt['pk_prob'])
    meta={'fold':fold,'best_meta':best_meta,'plain_metrics':plain_m,'tta_metrics':tta_m,'history':history}
    (fold_dir/'result.json').write_text(json.dumps(meta,indent=2))
    print('FOLD_RESULT',json.dumps({k:meta[k] for k in ['fold','best_meta','plain_metrics','tta_metrics']}),flush=True)

def smoke_test():
    seed_all(SEED)
    device=torch.device('cuda:0')
    model=Model().to(device).eval()
    x=torch.zeros(1,3,IMAGE_SIZE,IMAGE_SIZE,device=device,dtype=torch.float32)
    with torch.no_grad(), torch.autocast('cuda',dtype=torch.bfloat16):
        z=model.features(x)
    assert z.shape==(1, model.disaster.in_features), f'Unexpected feature shape: {tuple(z.shape)}'
    core=getattr(model.vision,'vision_model',model.vision)
    layers=getattr(getattr(core,'encoder',None),'layers',None)
    assert layers is not None and len(layers)>=UNFREEZE_LAST_N, 'Could not locate expected SigLIP vision transformer layers.'
    print(f'SMOKE_OK model={MODEL_ID} feature_dim={z.shape[-1]} layers={len(layers)} device={torch.cuda.get_device_name(0)}',flush=True)

def enumerate_test():
    rows=[]
    for p in sorted(TEST_DIR.iterdir(),key=lambda x:int(x.stem) if x.stem.isdigit() else x.name):
        if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.jfif','.bmp','.webp'}:
            rows.append({'path':str(p),'id':p.stem})
    df=pd.DataFrame(rows); df['row_index']=np.arange(len(df)); return df

def init_ddp():
    dist.init_process_group('nccl')
    rank=dist.get_rank(); local=int(os.environ['LOCAL_RANK']); world=dist.get_world_size()
    torch.cuda.set_device(local); return rank,local,world,torch.device(f'cuda:{local}')

def final_ddp(selection_path):
    rank,local,world,device=init_ddp(); seed_all(SEED+999+rank)
    sel=json.loads(Path(selection_path).read_text()); head_epochs=int(sel['head_epochs']); partial_epochs=int(sel['partial_epochs']); use_tta=bool(sel['use_tta'])
    assert FINAL_GLOBAL_BATCH%world==0
    per_gpu=FINAL_GLOBAL_BATCH//world
    df=pd.read_csv(MANIFEST); df['row_index']=np.arange(len(df))
    model=Model().to(device)
    stages=[('heads',head_epochs)] + ([('partial',partial_epochs)] if partial_epochs>0 else [])
    for stage,n_epochs in stages:
        set_stage(model,stage)
        # Re-wrap at each stage because trainable parameter set changes.
        ddp=DDP(model,device_ids=[local],output_device=local,broadcast_buffers=False,find_unused_parameters=True)
        sampler=DistributedSampler(ImgDS(df,True,True),num_replicas=world,rank=rank,shuffle=True,seed=SEED,drop_last=False)
        # construct loader from same dataset used by sampler
        ds=sampler.dataset
        ld=DataLoader(ds,batch_size=per_gpu,sampler=sampler,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
        opt=make_optimizer(model,stage); sch=cosine_sched(opt,len(ld)*n_epochs)
        for ep in range(1,n_epochs+1):
            sampler.set_epoch(ep + (0 if stage=='heads' else 100))
            t=time.time(); loss=train_epoch(ddp,ld,opt,sch,device,distributed=True)
            if rank==0: print(f'FULL_DDP {stage} {ep}/{n_epochs} loss={loss:.4f} min={(time.time()-t)/60:.1f}',flush=True)
        del ddp; torch.cuda.empty_cache(); dist.barrier()
    final_path=OUT/'full_train_final.pt'
    if rank==0:
        torch.save({'model':model.state_dict(),'selection':sel},final_path)
        print('Saved',final_path,flush=True)
    dist.barrier()
    if rank==0:
        # TEST inference on rank 0 only after final TRAIN completes.
        test=enumerate_test(); l0=loader(test,False,False,EVAL_BATCH,flip=False); p0=predict(model,l0,device)
        probs_j=p0['pj_prob']; probs_k=p0['pk_prob']
        if use_tta:
            l1=loader(test,False,False,EVAL_BATCH,flip=True); p1=predict(model,l1,device)
            probs_j=(probs_j+p1['pj_prob'])/2; probs_k=(probs_k+p1['pk_prob'])/2
        pred_j=probs_j.argmax(1); pred_k=probs_k.argmax(1)
        # Build rows from sample solution order, preserving ID strings exactly.
        sol=pd.read_csv(SOLUTION_PATH,sep=None,engine='python')
        out=sol.copy(); target=[]
        by_id={str(test.iloc[i].id):(SUB_JENIS[IDX_TO_JENIS[int(pred_j[i])]],SUB_KER[IDX_TO_KER[int(pred_k[i])]]) for i in range(len(test))}
        for rid in out['ID'].astype(str):
            base,suffix=rid.rsplit('_',1)
            assert base in by_id, f'Missing test ID {base}'
            target.append(by_id[base][0] if suffix=='jenis' else by_id[base][1])
        out['Target']=target
        out.to_csv(OUT/'submission.csv',index=False)
        np.savez_compressed(OUT/'test_probabilities.npz',ids=test.id.astype(str).to_numpy(),jenis_prob=probs_j,kerusakan_prob=probs_k)
        print(out.head(10).to_string(index=False)); print('Saved',OUT/'submission.csv',flush=True)
    dist.barrier(); dist.destroy_process_group()

if __name__=='__main__':
    ap=argparse.ArgumentParser(); ap.add_argument('--mode',choices=['smoke','fold','final'],required=True); ap.add_argument('--fold',type=int); ap.add_argument('--selection')
    a=ap.parse_args()
    if a.mode=='smoke':
        smoke_test()
    elif a.mode=='fold':
        assert a.fold is not None; fit_fold(a.fold)
    else:
        assert a.selection; final_ddp(a.selection)
'''
RUNNER_PATH.write_text(RUNNER_CODE)
os.chmod(RUNNER_PATH, 0o755)
print('Wrote:', RUNNER_PATH)


Wrote: /workspace/output/siglip2_b384_condsev_4gpu/train_runner.py


## Pre-training model-loader smoke test

Loads the FixRes SigLIP2 checkpoint through the compatible `SiglipVisionModel` class and runs one 384×384 dummy forward on GPU 0 before launching any CV jobs.


In [8]:
env=os.environ.copy()
env['CUDA_VISIBLE_DEVICES']='0'
env['HF_HUB_OFFLINE']='1'
env['TRANSFORMERS_OFFLINE']='1'
p=subprocess.run([sys.executable,str(RUNNER_PATH),'--mode','smoke'],env=env,text=True,capture_output=True)
print(p.stdout)
if p.returncode!=0:
    print(p.stderr)
    raise RuntimeError('SigLIP model-loader smoke test failed. Fix this before launching CV.')


SMOKE_OK model=google/siglip2-base-patch16-384 feature_dim=768 layers=12 device=NVIDIA GeForce RTX 5090



## Parallel 5-fold CV across 4 GPUs

Four folds start immediately on GPUs 0–3. Fold 4 is dispatched to the first GPU that becomes free. Each fold remains a normal single-GPU job with batch size 32, so the statistical training recipe is unchanged.

In [9]:
from concurrent.futures import ThreadPoolExecutor, as_completed

LOG_DIR = OUTPUT_DIR/'logs'; LOG_DIR.mkdir(exist_ok=True)

def run_fold_gpu(fold,gpu):
    env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=str(gpu); env['HF_HUB_OFFLINE']='1'; env['TRANSFORMERS_OFFLINE']='1'
    log_path=LOG_DIR/f'fold_{fold}_gpu{gpu}.log'
    with open(log_path,'w') as log:
        p=subprocess.run([sys.executable,str(RUNNER_PATH),'--mode','fold','--fold',str(fold)],env=env,stdout=log,stderr=subprocess.STDOUT)
    if p.returncode!=0:
        tail='\n'.join(log_path.read_text(errors='ignore').splitlines()[-80:])
        raise RuntimeError(f'Fold {fold} failed on GPU {gpu}. Tail:\n{tail}')
    print(f'Fold {fold} finished on GPU {gpu} | {log_path}')
    return gpu

# Skip completed folds to make the cell restart-safe.
pending=[f for f in range(5) if not (OUTPUT_DIR/f'fold_{f}'/'oof.npz').exists()]
print('pending folds:',pending)

initial=pending[:4]
remaining=pending[4:]
with ThreadPoolExecutor(max_workers=4) as ex:
    futures={ex.submit(run_fold_gpu,f,g):f for g,f in enumerate(initial)}
    if remaining:
        # Dispatch fold 4 to the first GPU that completes.
        first=next(as_completed(futures)); free_gpu=first.result(); finished_fold=futures[first]
        print('first completed:',finished_fold,'free GPU:',free_gpu)
        f=remaining.pop(0); futures[ex.submit(run_fold_gpu,f,free_gpu)]=f
    for fut in as_completed(list(futures)):
        fut.result()
print('All requested folds completed.')

pending folds: [0, 1, 2, 3, 4]
Fold 3 finished on GPU 3 | /workspace/output/siglip2_b384_condsev_4gpu/logs/fold_3_gpu3.log
first completed: 3 free GPU: 3
Fold 0 finished on GPU 0 | /workspace/output/siglip2_b384_condsev_4gpu/logs/fold_0_gpu0.log
Fold 2 finished on GPU 2 | /workspace/output/siglip2_b384_condsev_4gpu/logs/fold_2_gpu2.log
Fold 1 finished on GPU 1 | /workspace/output/siglip2_b384_condsev_4gpu/logs/fold_1_gpu1.log
Fold 4 finished on GPU 3 | /workspace/output/siglip2_b384_condsev_4gpu/logs/fold_4_gpu3.log
All requested folds completed.


## Pool OOF predictions and select the final TRAIN-only recipe

TTA is accepted only when pooled 5-fold OOF Micro F1 improves. Final partial-FT epoch count is the median fold-selected partial epoch, making it robust to one noisy fold.

In [10]:
from sklearn.metrics import f1_score, accuracy_score

all_rows=[]; metas=[]
for f in range(5):
    z=np.load(OUTPUT_DIR/f'fold_{f}'/'oof.npz')
    d={k:z[k] for k in z.files}; all_rows.append(d)
    metas.append(json.loads((OUTPUT_DIR/f'fold_{f}'/'result.json').read_text()))

idx=np.concatenate([d['row_index'] for d in all_rows]).astype(int)
assert len(idx)==len(manifest) and len(np.unique(idx))==len(manifest), 'OOF coverage must be exactly once per TRAIN image.'
order=np.argsort(idx)
yj=np.concatenate([d['yj'] for d in all_rows])[order]
yk=np.concatenate([d['yk'] for d in all_rows])[order]
plain_j=np.concatenate([d['plain_j'] for d in all_rows])[order]
plain_k=np.concatenate([d['plain_k'] for d in all_rows])[order]
tta_j=np.concatenate([d['tta_j'] for d in all_rows])[order]
tta_k=np.concatenate([d['tta_k'] for d in all_rows])[order]

def pooled_metrics(jp,kp):
    pj=jp.argmax(1); pk=kp.argmax(1)
    yt=np.concatenate([yj,yk]); yp=np.concatenate([pj,pk])
    return {
        'micro_f1':float(f1_score(yt,yp,average='micro')),
        'acc_jenis':float(accuracy_score(yj,pj)),
        'acc_kerusakan':float(accuracy_score(yk,pk)),
    }

PLAIN=pooled_metrics(plain_j,plain_k); TTA=pooled_metrics(tta_j,tta_k)
print('POOLED PLAIN:',PLAIN)
print('POOLED TTA  :',TTA)

cv_table=pd.DataFrame([{ 
    'fold':m['fold'], 'best_stage':m['best_meta']['stage'], 'best_stage_epoch':m['best_meta']['stage_epoch'],
    'plain_micro_f1':m['plain_metrics']['micro_f1'], 'tta_micro_f1':m['tta_metrics']['micro_f1']
} for m in metas]).sort_values('fold')
display(cv_table)
cv_table.to_csv(OUTPUT_DIR/'cv_summary.csv',index=False)

partial_epochs=[int(m['best_meta']['stage_epoch']) if m['best_meta']['stage']=='partial' else 0 for m in metas]
FINAL_PARTIAL_EPOCHS=int(np.median(partial_epochs))
FINAL_USE_TTA=bool(TTA['micro_f1'] > PLAIN['micro_f1'])
selection={
    'head_epochs':2,
    'partial_epochs':FINAL_PARTIAL_EPOCHS,
    'use_tta':FINAL_USE_TTA,
    'pooled_plain':PLAIN,
    'pooled_tta':TTA,
}
SELECTION_PATH=OUTPUT_DIR/'final_selection.json'; SELECTION_PATH.write_text(json.dumps(selection,indent=2))
print(json.dumps(selection,indent=2))

POOLED PLAIN: {'micro_f1': 0.9278114632193113, 'acc_jenis': 0.9954810662395607, 'acc_kerusakan': 0.8601418601990619}
POOLED TTA  : {'micro_f1': 0.9277828623727262, 'acc_jenis': 0.9951950577737101, 'acc_kerusakan': 0.8603706669717424}


,fold,best_stage,best_stage_epoch,plain_micro_f1,tta_micro_f1
0,0,partial,4,0.926079,0.925365
1,1,partial,3,0.927059,0.926773
2,2,partial,4,0.930655,0.930941
3,3,partial,5,0.928633,0.928490
4,4,partial,4,0.926630,0.927346


{
  "head_epochs": 2,
  "partial_epochs": 4,
  "use_tta": false,
  "pooled_plain": {
    "micro_f1": 0.9278114632193113,
    "acc_jenis": 0.9954810662395607,
    "acc_kerusakan": 0.8601418601990619
  },
  "pooled_tta": {
    "micro_f1": 0.9277828623727262,
    "acc_jenis": 0.9951950577737101,
    "acc_kerusakan": 0.8603706669717424
  }
}


## Final retraining on 100% TRAIN with 4-GPU DDP

The final model uses one process per GPU. Per-GPU batch is 8, so `8 × 4 = 32` global batch, matching the original single-GPU recipe. LR values are therefore intentionally unchanged.

In [13]:
import os
import sys
import shlex
import subprocess
from pathlib import Path

# Explicit paths so this cell can run independently
RUNNER_PATH = Path("/workspace/output/siglip2_b384_condsev_4gpu/train_runner.py")
SEL_PATH = Path("/workspace/output/siglip2_b384_condsev_4gpu/final_selection.json")
LOG_DIR = Path("/workspace/output/siglip2_b384_condsev_4gpu/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

assert RUNNER_PATH.exists(), f"Runner not found: {RUNNER_PATH}"
assert SEL_PATH.exists(), f"Selection file not found: {SEL_PATH}"

cmd = [
    sys.executable,
    "-m", "torch.distributed.run",
    "--standalone",
    "--nproc_per_node=4",
    str(RUNNER_PATH),
    "--mode", "final",
    "--selection", str(SEL_PATH),
]

print("Launching:", " ".join(shlex.quote(x) for x in cmd))

log_path = LOG_DIR / "final_ddp.log"

ddp_env = os.environ.copy()
ddp_env["HF_HUB_OFFLINE"] = "1"
ddp_env["TRANSFORMERS_OFFLINE"] = "1"

with open(log_path, "w") as log:
    p = subprocess.Popen(
        cmd,
        env=ddp_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in p.stdout:
        print(line, end="")
        log.write(line)
        log.flush()

    rc = p.wait()

if rc != 0:
    tail = "\n".join(
        log_path.read_text(errors="ignore").splitlines()[-120:]
    )
    raise RuntimeError(
        f"Final 4-GPU DDP training failed with exit code {rc}.\n"
        f"Log: {log_path}\n\n"
        f"Last 120 lines:\n{tail}"
    )

print("\nFinal 4-GPU DDP training completed.")
print("Log:", log_path)

Launching: /venv/main/bin/python -m torch.distributed.run --standalone --nproc_per_node=4 /workspace/output/siglip2_b384_condsev_4gpu/train_runner.py --mode final --selection /workspace/output/siglip2_b384_condsev_4gpu/final_selection.json

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behav

## Submission sanity checks

In [14]:
submission=pd.read_csv(OUTPUT_DIR/'submission.csv')
print(submission.head(12).to_string(index=False))
print('\nshape:',submission.shape)
print('columns:',submission.columns.tolist())
print('Target counts:')
print(submission.Target.value_counts(dropna=False).sort_index())
assert submission.columns.tolist()==['ID','Target']
assert len(submission)==900, f'Expected 900 rows, got {len(submission)}'
assert submission.Target.notna().all()
assert set(submission.Target.astype(int).unique()).issubset({1,2,3})
print('\nSubmission ready:', OUTPUT_DIR/'submission.csv')

         ID  Target
    1_jenis       1
1_kerusakan       1
    2_jenis       1
2_kerusakan       1
    3_jenis       1
3_kerusakan       1
    4_jenis       1
4_kerusakan       1
    5_jenis       1
5_kerusakan       1
    6_jenis       1
6_kerusakan       2

shape: (900, 2)
columns: ['ID', 'Target']
Target counts:
Target
1    324
2    318
3    258
Name: count, dtype: int64

Submission ready: /workspace/output/siglip2_b384_condsev_4gpu/submission.csv


## Artifacts

```text
/workspace/output/siglip2_b384_condsev_4gpu/
├── train_hashes.csv
├── train_manifest_5fold.csv
├── train_runner.py
├── fold_0/ ... fold_4/
│   ├── best.pt
│   ├── oof.npz
│   └── result.json
├── cv_summary.csv
├── final_selection.json
├── full_train_final.pt
├── test_probabilities.npz
├── submission.csv
└── logs/
```

The final submission remains one model trained on 100% TRAIN. The four GPUs are used to make validation substantially stronger and final training faster, not to change the architecture or introduce a seed/model ensemble.